In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import time

# ==========================================
# 1. TRANSFER FONKSİYONU (Continuous -> Binary)
# ==========================================
def sigmoid_transfer(x):
    """Sürekli değerleri [0, 1] aralığında olasılığa dönüştürür."""
    x = np.clip(x, -10, 10) # exp overflow hatasını önlemek için kırpma
    return 1 / (1 + np.exp(-x))

# ==========================================
# 2. PIECEWISE CHAOS MAPPING (Başlangıç Popülasyonu)
# ==========================================
def piecewise_chaos_initialization(pop_size, dim):
    """Makaledeki Parçalı Kaos Haritalaması ile popülasyon çeşitliliğini artırır."""
    P = 0.4
    X_chaos = np.zeros((pop_size, dim))
    val = np.random.rand(dim)
    for i in range(pop_size):
        for j in range(dim):
            if 0 <= val[j] < P:
                val[j] = val[j] / P
            elif P <= val[j] < 0.5:
                val[j] = (val[j] - P) / (0.5 - P)
            elif 0.5 <= val[j] < 1 - P:
                val[j] = (1 - P - val[j]) / (0.5 - P)
            else:
                val[j] = (1 - val[j]) / P
        X_chaos[i, :] = val * 10 - 5 # Değerleri [-5, 5] uzayına ölçekle
    return X_chaos

# ==========================================
# 3. FITNESS (AMAÇ) FONKSİYONU
# ==========================================
def evaluate_fitness(selected_indices, X_train, X_test, y_train, y_test, alpha=0.99, beta=0.01):
    """RMSE ve Seçilen Özellik sayısına göre modelin uygunluğunu ölçer."""
    if np.sum(selected_indices) == 0:
        return float('inf'), float('inf') # Geçersiz çözüm (hiç özellik seçilmediyse)
        
    X_train_sel = X_train.iloc[:, selected_indices == 1]
    X_test_sel = X_test.iloc[:, selected_indices == 1]
    
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train_sel, y_train)
    predictions = model.predict(X_test_sel)
    
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    total_features = X_train.shape[1]
    selected_count = np.sum(selected_indices)
    
    # Standart ağırlıklar: %99 Hata + %1 Boyut Azaltma
    fitness = (alpha * rmse) + (beta * (selected_count / total_features))
    return fitness, rmse

# ==========================================
# 4. Binary MPSOGOA ALGORİTMASI 
# ==========================================
def run_bMPSOGOA(X_train, X_test, y_train, y_test, pop_size=10, max_iter=20):
    total_features = X_train.shape[1]
    
    # Adım 1: Kaos Haritalaması ile Başlatma
    X = piecewise_chaos_initialization(pop_size, total_features)
    V = np.zeros((pop_size, total_features)) # Hız matrisi (PSO)
    
    pbest_X = np.copy(X)
    pbest_fitness = np.full(pop_size, float('inf'))
    
    gbest_X = np.zeros(total_features)
    gbest_fitness = float('inf')
    gbest_rmse = float('inf')
    
    # PSO Katsayıları
    w_max, w_min = 0.9, 0.4
    c1, c2 = 1.5, 1.5
    
    for t in range(max_iter):
        w = w_max - t * ((w_max - w_min) / max_iter) # Adaptif Atalet Ağırlığı
        
        for i in range(pop_size):
            # Adım 2: Transfer Fonksiyonu ile Binary Dönüşüm
            prob = sigmoid_transfer(X[i])
            X_bin = (prob > 0.5).astype(int)
            
            if np.sum(X_bin) == 0:
                X_bin[np.random.randint(0, total_features)] = 1
                
            # Değerlendirme ve Güncelleme
            fitness, rmse = evaluate_fitness(X_bin, X_train, X_test, y_train, y_test)
            
            if fitness < pbest_fitness[i]:
                pbest_fitness[i] = fitness
                pbest_X[i] = np.copy(X[i])
                
            if fitness < gbest_fitness:
                gbest_fitness = fitness
                gbest_X = np.copy(X[i])
                gbest_rmse = rmse

        # Adım 3: MPSOGOA Matematiksel Pozisyon Güncellemesi
        for i in range(pop_size):
            r1, r2, r3 = np.random.rand(), np.random.rand(), np.random.rand()
            
            # PSO Hız (Velocity) Güncellemesi
            V[i] = w * V[i] + c1 * r1 * (pbest_X[i] - X[i]) + c2 * r2 * (gbest_X - X[i])
            
            # GOA Hibrit Dinamikleri (Ceylan Otlama / Yırtıcıdan Kaçış)
            if r3 < 0.5:
                # Sömürü (Exploitation): Otlama davranışı
                X[i] = X[i] + V[i] * np.random.normal(0, 1, total_features)
            else:
                # Keşif (Exploration): Sıçrama (Levy Walk benzeri)
                levy_step = np.random.standard_cauchy(total_features)
                X[i] = X[i] + V[i] + (gbest_X - X[i]) * levy_step
                
            # Adım 4: Global Pertürbasyon (Makaledeki yerel minimumdan kaçış taktiği)
            if np.random.rand() < 0.1:
                X[i] = X[i] + np.random.uniform(-1, 1, total_features)
                
    # Döngü sonu, en iyi sonucu binary olarak çıkar
    best_prob = sigmoid_transfer(gbest_X)
    best_bin = (best_prob > 0.5).astype(int)
    if np.sum(best_bin) == 0:
        best_bin[0] = 1
        
    return np.sum(best_bin), gbest_rmse

# ==========================================
# 5. ANA ÇALIŞTIRMA VE TEST (10 Çalıştırma)
# ==========================================
file_path = 'energydata_complete.csv'
df = pd.read_csv(file_path)

X = df.drop(columns=['date', 'Appliances'])
y = df['Appliances']
total_features = X.shape[1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = []

print("Starting 10 simulation runs with bMPSOGOA...\n")
print(f"{'Run':<5} | {'Original Features':<18} | {'Selected Features':<17} | {'RMSE':<12} | {'Time (s)':<10}")
print("-" * 75)

for i in range(1, 11):
    start_time = time.time()
    
    # Her çalıştırmada küçük bir popülasyonla kısa iterasyonlar atıyoruz (hız için)
    selected_feature_count, rmse = run_bMPSOGOA(X_train, X_test, y_train, y_test, pop_size=10, max_iter=10)
    
    elapsed_time = round(time.time() - start_time, 2)
    results.append([i, total_features, selected_feature_count, round(rmse, 4), elapsed_time])
    
    print(f"{i:<5} | {total_features:<18} | {selected_feature_count:<17} | {round(rmse, 4):<12} | {elapsed_time:<10}")

print("-" * 75)
df_results = pd.DataFrame(results, columns=['Run', 'Original Features', 'Selected Features', 'RMSE', 'Time (s)'])
print(f"AVERAGE: {df_results['Selected Features'].mean()} features with {df_results['RMSE'].mean():.4f} RMSE")

Starting 10 simulation runs...

Run   | Original Features  | Selected Features | RMSE         | Time (s)  
---------------------------------------------------------------------------
1     | 27                 | 18                | 96.3057      | 0.49      
2     | 27                 | 18                | 92.4534      | 0.47      
3     | 27                 | 20                | 95.0123      | 0.55      
4     | 27                 | 20                | 95.4952      | 0.53      
5     | 27                 | 20                | 97.0211      | 0.56      
6     | 27                 | 20                | 89.7234      | 0.57      
7     | 27                 | 18                | 97.1826      | 0.51      
8     | 27                 | 16                | 94.8777      | 0.4       
9     | 27                 | 20                | 96.8871      | 0.52      
10    | 27                 | 16                | 86.3201      | 0.41      
-------------------------------------------------------------------